In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/wustl-iiot-2021-dataset/wustl_iiot_2021.csv
/kaggle/input/wustl-iiot-2021-updated/wustl_iiot_2021.csv
/kaggle/input/wustl-iiot-2021-updated/balanced_train_data_SMOTEENNLOF.npz
/kaggle/input/wustl-iiot-2021-updated/WUSTL_Preprocessing.ipynb
/kaggle/input/wustl-iiot-2021-updated/Unbal_train_data.npz
/kaggle/input/wustl-iiot-2021-updated/test_data.npz
/kaggle/input/wustl-iiot-2021-updated/wustl_corrected.csv


In [3]:
# ======================================
# CELL 1 – IMPORT LIBRARIES
# ======================================

import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier

import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")


/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Libraries Imported Successfully


In [4]:
# ======================================
# CELL 2 – DATA LOADING & PREPROCESSING
# ======================================

# Load dataset (Kaggle path)
df = pd.read_csv("/kaggle/input/wustl-iiot-2021-dataset/wustl_iiot_2021.csv")

print("Original Shape:", df.shape)
print("Columns:", df.columns)

# Remove identifier columns
cols_to_drop = [
    'StartTime', 'LastTime', 'SrcAddr', 'DstAddr',
    'Sport', 'Dport', 'Proto', 'Dir', 'state'
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

print("After Dropping Identifier Columns:", df.shape)

# Encode MULTI-CLASS target (Traffic)
le = LabelEncoder()
df['Traffic'] = le.fit_transform(df['Traffic'])

# Drop binary column (Target)
X = df.drop(['Traffic', 'Target'], axis=1)
y = df['Traffic']

print("Number of Classes:", y.nunique())

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Original Shape: (1194464, 49)
Columns: Index(['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport',
       'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
       'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
       'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt',
       'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max',
       'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes',
       'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct',
       'Traffic', 'Target'],
      dtype='object')
After Dropping Identifier Columns: (1194464, 42)
Number of Classes: 5
Training Samples: 955571
Testing Samples: 238893


In [5]:
# ======================================
# CELL 3 – TRAIN BASE MODELS
# ======================================

num_classes = y_train.nunique()
print("Number of classes:", num_classes)

# -------------------------------
# XGBoost (Multi-class)
# -------------------------------
xgb = XGBClassifier(
    n_estimators=300,
    objective="multi:softmax",   # ✅ SHAP-safe objective
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train, y_train)

# -------------------------------
# LightGBM
# -------------------------------
lgb = LGBMClassifier(
    n_estimators=300,
    objective="multiclass",
    num_class=num_classes,
    random_state=42
)

lgb.fit(X_train, y_train)

# -------------------------------
# CatBoost
# -------------------------------
cat = CatBoostClassifier(
    n_estimators=300,
    loss_function="MultiClass",
    verbose=0,
    random_state=42
)

cat.fit(X_train, y_train)

print("All three models trained successfully.")


Number of classes: 5
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.113445 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7253
[LightGBM] [Info] Number of data points in the train set: 955571, number of used features: 40
[LightGBM] [Info] Start training from score -8.634266
[LightGBM] [Info] Start training from score -8.437346
[LightGBM] [Info] Start training from score -2.724841
[LightGBM] [Info] Start training from score -4.976452
[LightGBM] [Info] Start training from score -0.075640
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [7]:
# ======================================
# CELL 4 – SHAP FEATURE WEIGHT EXTRACTION (KAGGLE SAFE)
# ======================================

import shap

# Take small background sample
background = X_train.sample(200, random_state=42)

# Use unified explainer with explicit model_output
explainer = shap.Explainer(xgb.predict, background)

# Explain small sample
sample_data = X_train.sample(300, random_state=42)
shap_values = explainer(sample_data)

# shap_values.values shape: (samples, features)
feature_importance = np.mean(np.abs(shap_values.values), axis=0)

# Normalize weights
feature_weights = feature_importance / np.max(feature_importance)

print("Dynamic feature weights extracted successfully.")


PermutationExplainer explainer: 301it [01:37,  2.85it/s]                         

Dynamic feature weights extracted successfully.


In [8]:
# ======================================
# CELL 5 – APPLY FEATURE WEIGHTING
# ======================================

# Convert weights to numpy array
weights_vector = np.array(feature_weights)

# Apply dynamic weighting
X_train_w = X_train * weights_vector
X_test_w = X_test * weights_vector

print("Dynamic feature weighting applied.")


# ======================================
# SOFT VOTING ENSEMBLE
# ======================================

from sklearn.ensemble import VotingClassifier

hybrid_model = VotingClassifier(
    estimators=[
        ('xgb', xgb),
        ('lgb', lgb),
        ('cat', cat)
    ],
    voting='soft'   # 🔥 Soft voting (probability averaging)
)

# Train ensemble on weighted data
hybrid_model.fit(X_train_w, y_train)

# Predict
preds = hybrid_model.predict(X_test_w)

print("Hybrid Soft Voting Ensemble trained successfully.")



Dynamic feature weighting applied.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.283861 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6993
[LightGBM] [Info] Number of data points in the train set: 955571, number of used features: 37
[LightGBM] [Info] Start training from score -8.634266
[LightGBM] [Info] Start training from score -8.437346
[LightGBM] [Info] Start training from score -2.724841
[LightGBM] [Info] Start training from score -4.976452
[LightGBM] [Info] Start training from score -0.075640
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

In [16]:
# PERFORMANCE METRICS
# ======================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds, average='weighted')
recall = recall_score(y_test, preds, average='weighted')
f1 = f1_score(y_test, preds, average='weighted')

print("\n========== FINAL PERFORMANCE ==========")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-Score :", f1)

print("\n========== CLASSIFICATION REPORT ==========\n")
print(classification_report(y_test, preds))

# ======================================
# TRAINING vs TESTING ACCURACY
# ======================================

# Training predictions
train_pred = hybrid_model.predict(X_train_w)

# Training accuracy
train_accuracy = accuracy_score(y_train, train_pred)

# Testing accuracy
test_accuracy = accuracy_score(y_test, preds)

print("Training Accuracy :", train_accuracy)
print("Testing Accuracy  :", test_accuracy)




========== FINAL PERFORMANCE ==========
Accuracy : 0.9999623262297347
Precision: 0.9999645100463219
Recall   : 0.9999623262297347
F1-Score : 0.9999630865532314

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

           0       0.87      0.95      0.91        42
           1       1.00      1.00      1.00        52
           2       1.00      1.00      1.00     15661
           3       1.00      1.00      1.00      1648
           4       1.00      1.00      1.00    221490

    accuracy                           1.00    238893
   macro avg       0.97      0.99      0.98    238893
weighted avg       1.00      1.00      1.00    238893

Training Accuracy : 0.9999895350528637
Testing Accuracy  : 0.9999623262297347
